# Cars 4 You - Deployment Notebook

## Kaggle Competition Submission

This notebook loads the trained model and generates predictions for the test dataset.
It replicates all preprocessing steps from the EDA notebook to ensure consistency.

## Import Packages

In [1]:
import os
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')
from rapidfuzz import process, fuzz
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

## Define Custom Transformers

In [2]:
class RareLabelGrouper(BaseEstimator, TransformerMixin):
    
    """
    This class will be useful to identify and group rare instances in the model column as "other". 
    The minimal proportion needed to not be grouped is min_prop.
    
    """
   
    def __init__(self, min_prop=0.005, column=None): 
        self.min_prop = min_prop
        
        self.column = column
        
        self.keep_values_ = None  #This will be used to store the values to group
    
    
    def fit(self, X, y=None):
        s = X if isinstance(X, pd.Series) else pd.Series(X.iloc[:,0] if hasattr(X, "iloc") else X[:,0])  # 
        
        vc = s.value_counts(dropna=False) 
        
        total = len(s)  
        
        keep = vc[vc / total >= self.min_prop].index # Chooses the categories for which the minimum proportion is met
        
        self.keep_values_ = set(keep.tolist()) # Attributes them to an attribute of the class in the form of a list
        
        return self
    
    
    def transform(self, X):
        """
        np.where() works like an if-else statement. If s is in stored values to keep, then keep s, else attribute "Other".
        To attribute the name of the column, if it was passed and stored in the attribute .column, then use that name, else use the generic 'col' name.
        """
        s = X if isinstance(X, pd.Series) else pd.Series(X.iloc[:,0] if hasattr(X, "iloc") else X[:,0])  #Same pandas series problem
        return pd.DataFrame(np.where(s.isin(self.keep_values_), s, "Other"),  
                            columns=[self.column if self.column else "col"])

    
    def get_feature_names_out(self, input_features=None):

        if input_features is not None:
            return np.asarray(input_features, dtype=object)

        col_name = self.column if self.column else "col"
        return np.asarray([col_name], dtype=object)


In [3]:
class Mode_Imput_By_Brand (BaseEstimator, TransformerMixin):

    """
    O objetivo com este transformer é preencher os valores nulos de cada coluna através da moda, mas primeiro, agrupar por brand.
    Como fallback, caso nao haja nenhuma linha preenchida daquela brand em especifico, depois de utilizar este transformer
    iremos utilzar o SimpleImputer.
    
    """
    
    def __init__(self, imput_cols = None , group_col = "brand"):       
        self.imput_cols = imput_cols  #Store the imputing values for each group
        self.group_col = group_col #Stores the variable name in which we want to group to input the value we need (for example, group by the brand to input the most likely model of a given car)

    # é obrigatorio colocar X e y em todos os metodos de fit do sklearn
    def fit(self , X, y = None):
        X = X.copy()  # em qualquer metodo do sklearn criamos uma copia do df original, pois no pipeline é usado o mesmo df nao modificado para os vários passos
        
        if self.imput_cols is None: # sklearn nao gosta de args obrigatórios entao fazemos isto para contornar
            self.imput_cols = [col for col in X.columns if col != self.group_col] #Iterates over all columns that are not group_col and stores their names in a list 
        
        self.mode_maps_ = {}

        for col in self.imput_cols:  #Iterate over all columns except the group_col
            mode_map = X.groupby(self.group_col)[col].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)  #Calculates the mode for each group and if there isn't assume np.nan
            self.mode_maps_[col] = mode_map

        return self

        
    def transform(self, X):
        X = X.copy()

        for col in self.imput_cols:  #Iterate over all columns except the group_col
            X[col] = X[col].fillna(X[self.group_col].map(self.mode_maps_[col]))  #Fills all nans in one column (col) with the mode found in the mode_maps_ dict, which contains the mode for the different groups that were grouped through the group_col

        return X

In [4]:
class Imput_By_Brand_and_Model (BaseEstimator, TransformerMixin):

    """

    The goal with this transformer is to fullfill the Nans by the mode,  but 1st grouping by brand and model. Has a fallback
    in the pipeline, after using this transformer, it should be used a SimplImputer to guarantee that in the end, there is no missing
    values.

    =============================================================================================================================

    Parameters:

        imput_cols: List of cols you want to imput, if none is passed then it will assume all columns should be imputed besides the 
        ones on the group_cols

        group_cols: List of cols you want to groupby, it's defaulted to be brand and model

        .mode_maps: Dictionary for each imput column, where each key is an imput_col and each value is a dictionary where each key is
                    a combiantion of brand and model and each value is the mode

    ==============================================================================================================================

    Methods:
    
        .fit: Creates a loop that for each column to imput: 0. Creates an empty dict to store each imput col as a key
                                                            1. Groups the data by the group_cols
                                                            2. Calculates the mode, if more than 1 get´s the 1st one 
                                                            if there is no value it returns Nan
                                                            3. Stores the combination of brand and model (group_cols) and mode
                                                            as a key-value pair 
        
        .transform: 1. For each row of the dataset stores the pair of (brand,model) into a Series
                    2. Loops trough the imput_cols and for each one it uses the pairs created before as a key to get the value of 
                    the mode_maps, storing this value in a Series called mapped
                    3. Goes trough each line of the data, if the value is nan it fulfiles it with the corresponding value of the
                    mapped series

    ==============================================================================================================================

    Attributes:

        mode_maps_ : Dictionary of dictionarys that stores the mode for each pair in each column of X.

                     {{"feature_1:"("bmw","i8"): mode,
                     ('vw', 'up!'): mode},
                     {"feature_2": "("bmw","i8"): mode,
                     ('vw', 'up!'): mode}}
                     
    
    """

    def __init__(self, imput_cols = None , group_cols = ["brand", "model"]):       
        self.imput_cols = imput_cols
        self.group_cols = group_cols

    
    def fit(self , X, y = None):
        X = X.copy()  
        
        if self.imput_cols is None: 
            self.imput_cols = [col for col in X.columns if col != self.group_cols]
        
        self.mode_maps_ = {}

        for col in self.imput_cols:
            mode_map = X.groupby(self.group_cols)[col].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan) 
            self.mode_maps_[col] = mode_map.to_dict()

        return self 
    
    def transform(self, X):
        X = X.copy()

        keys = list(zip(*[X[g] for g in self.group_cols]))  # [(brand, model), ...]
        key_series = pd.Series(keys, index=X.index)

        for col in self.imput_cols:
            mapped = key_series.map(self.mode_maps_[col])
            X[col] = X[col].fillna(mapped)

        return X

In [5]:
class Log1pSkewed(BaseEstimator, TransformerMixin):
    """ 
    This transformer computes the natural logarithm of (1 + x) for the columns specified in log_candidates. 
    The goal is to reduce right-skewness and stabilise variance in those features.
     ========================================================================================================

    Parameters: 
        numeric_cols: list of str
                      Names of the numeric columns of X
        log_candidates: list of str
                        Names of the columns of X that are right_skewed
     ========================================================================================================

    Methods:
        .fit: Stores the name of all features the indices of the features that will
        undergo through the log1p transformation (log_candidates).
        .transform: Applies the log1p transformation to the selected columns.
        .get_feature_names_out:        
    ========================================================================================================

    Attributes:

        n_features_in_: number of features before applying the fit
        feature_names_in_: name of the features before applying the fit
        log_idx_: indices of the columns in log_candidates
    """
    
    def __init__(self, numeric_cols, log_candidates):
        self.numeric_cols = numeric_cols
        self.log_candidates = log_candidates

    def fit(self, X, y=None):  
        #X = validate_data(self, X, accept_sparse=False, reset=True) # guarantees a correct format and stores n_features_in_
        #X = self._validate_data(X, accept_sparse=False, reset=True, force_all_finite='allow-nan') # guarantees a correct format and stores n_features_in_
        if hasattr(X, "columns"): # here the intention is to give more flexibility, on the current pipeline, we know that it will receive an array.
            self.feature_names_in_ = np.array(X.columns, dtype=object) 
        else:
            self.feature_names_in_ = np.array(self.numeric_cols, dtype=object)
        self.log_idx_ = [self.numeric_cols.index(c) for c in self.log_candidates]
        return self

    
    def transform(self, X):        
        X = np.asarray(X).copy() # turn X into an array just in case       
        X[:, self.log_idx_] = np.log1p(X[:, self.log_idx_])        
        return X

    
    def get_feature_names_out(self, input_features=None):
        
        if input_features is not None:
            return np.asarray(input_features, dtype=object)
        
        elif hasattr(self, "feature_names_in_"): # just in case this is the 1st transformer, even tough it probably never will
            return self.feature_names_in_ 

In [6]:
class ColumnSelector(BaseEstimator, TransformerMixin):
    """
    Selects a single column from the incoming DataFrame X, based on the column name provided at initialization.
    This column is returned as a DataFrame
    """
    
    def __init__(self, column): 
        self.column = column
    
    
    def fit(self, X, y=None): 
        return self
    
    
    def transform(self, X): 
        return pd.DataFrame(X[self.column])


    def get_feature_names_out(self, input_features=None):
        return np.asarray([self.column], dtype=object)

In [7]:
class PreprocessAndRemoveByName(BaseEstimator, TransformerMixin):

    """
    Wrapper transformer that applies a given preprocessing pipeline and removes a predefined set of output features by name.

    ========================================================================================================

    Parameters: 

        preprocessor: sklearn.pipeline
                    A preprocessing pipeline

        removed_features: list of str
                        list of undesired features

        feature_name_step: str, default="column_transformer"
                         Name of the step within the preprocessing pipeline from which output 
                         feature names are extracted.
                        

     ========================================================================================================

    Methods:

        .fit: Stores the name of all features the indices of the features that will
        undergo through the log1p transformation (log_candidates).

        .transform: Applies the log1p transformation to the selected columns.

        .get_feature_names_out: 
        
    ========================================================================================================

    Attributes:

        self.preprocessor_: sklearn.pipeline
                            Fitted clone of the preprocessing pipeline provided at initialization.
                            This instance is created during `fit` to avoid sharing fitted state
                            across cross-validation folds
                            

        keep_indices_ : list of int
                        Indices of the output features retained after preprocessing.
                        These indices correspond to the features whose names are not included
                        in 'removed_features' and are used to slice the transformed feature
                        matrix during 'transform'.


        removed_found_: list of str
                        Subset of `removed_features` that were found among the output features
                        of the preprocessing pipeline. This attribute is useful for debugging 
                        and verification of the feature-removal process.
    """

    
    def __init__(self, preprocessor, removed_features, feature_name_step="column_transformer"):
        self.preprocessor = preprocessor
        self.removed_features = removed_features
        self.feature_name_step = feature_name_step  # this is the step inside de preprocessor that has the names of the features, which will be "column_transformer" or any other name given to the ColumnTransformer

    def fit(self, X, y=None):
        # this prevents from using the same fitted preprocessor in different folds, in each fold it creates a clone of the "exterior" pipeline which in this case is PreprocessAndRemoveByName so we need to force a new clone of the pipeline inside it  
        self.preprocessor_ = clone(self.preprocessor) 
        self.preprocessor_.fit(X, y)

        # We go to the inside pipeline and to the specific step (Column Transformer name) to get the features names
        feature_names = self.preprocessor_.named_steps[self.feature_name_step].get_feature_names_out()
        feature_names = list(feature_names)

        self.keep_indices_ = [i for i, f in enumerate(feature_names) if f not in self.removed_features]

        self.feature_names_out_ = [feature_names[i] for i in self.keep_indices_] # update the fetures that will be kepted
        return self

    def transform(self, X):
        Xt = self.preprocessor_.transform(X)
        return Xt[:, self.keep_indices_]

    def get_feature_names_out(self, input_features=None):
        return np.array(self.feature_names_out_)

In [8]:
class ModelPredictorImputer(BaseEstimator, TransformerMixin):
    """
    Impute missing 'model' values using a classifier trained within each brand.
    Uses features like engineSize, mpg, transmission, fuelType to predict model.
    """

    def __init__(self, n_estimators=50):
        self.n_estimators = n_estimators
        self.brand_classifiers_ = {}
        self.brand_model_modes_ = {}
        self.brand_encoders_ = {}
        self.cat_cols = ['transmission', 'fuelType']
        self.num_cols = ['tax', 'mpg', 'engineSize', 'car_age']
        self.feature_cols = self.cat_cols + self.num_cols
        

    def fit(self, X, y=None):
        X = X.copy()
        for brand in X['brand'].dropna().unique():
            brand_data = X[X['brand'] == brand].copy()

            # Store mode as fallback
            if not brand_data['model'].mode().empty:
                self.brand_model_modes_[brand] = brand_data['model'].mode().iloc[0]           

            # Only train if we have enough complete cases
            complete_data = brand_data.dropna(subset=['model'] + self.feature_cols)
            if len(complete_data) >= 20 and complete_data['model'].nunique() > 1:                
                X_train = complete_data[self.feature_cols]
                y_train = complete_data['model']
                
                preprocessor_temp = ColumnTransformer([
                    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), 
                     self.cat_cols)],
                    remainder='passthrough'
                )
                
                pipeline_temp = Pipeline([
                    ('prep', preprocessor_temp),
                    ('clf', RandomForestClassifier(
                        n_estimators=self.n_estimators,
                        random_state=44,
                        n_jobs=-1
                    ))
                ])

                clf = RandomForestClassifier(
                    n_estimators=self.n_estimators,
                    random_state=44,
                    n_jobs=-1,
                    class_weight='balanced'
                )
                
                pipeline_temp.fit(X_train, y_train)
                self.brand_classifiers_[brand] = pipeline_temp
        
        return self
    

    def transform(self, X):
        X = X.copy()
        missing_model_idx = X[X['model'].isnull()].index     

        for idx in missing_model_idx:
            brand = X.loc[idx, 'brand']
    
            
            if pd.notna(brand) and brand in self.brand_classifiers_:
                features = X.loc[[idx], self.feature_cols]
                
                if not features.isnull().any().any():
                    try:
                        predicted_model = self.brand_classifiers_[brand].predict(features)[0]
                        X.loc[idx, 'model'] = predicted_model
                        continue
                    except Exception:
                        pass
                    
            # Fallback to brand mode            
            if pd.notna(brand) and brand in self.brand_model_modes_:
                X.loc[idx, 'model'] = self.brand_model_modes_[brand]
                continue
            
            X.loc[idx, 'model'] = self.global_mode_
        
        return X


## Define Helper Functions for Preprocessing

In [9]:
def remove_leading_space(s):
    if isinstance(s, str) and s.startswith(' '):
        return s[1:]
    return s

def rf_top1(query, choices, scorer=fuzz.ratio):
    match = process.extractOne(query, choices, scorer=scorer)
    if match is None:
        return np.nan, 0.0
    return match[0], match[1]

def match_model_in_make(model, brand_clean, brand):
    if pd.isna(brand):
        all_models = mod_by_fil_make
        best, sc = rf_top1(model, all_models, scorer=fuzz.ratio)
        if sc >= THRESH_MODEL_NO_BRAND:
            return best, sc
        else:
            return np.nan, 0.0
    if pd.isna(brand_clean):
        return np.nan, 0.0
    else:
        models_make = models_by_make[brand_clean]
        best, sc = rf_top1(model, models_make, scorer=fuzz.ratio) 
        if sc > THRESH_MODEL:
            return best, sc
        else:
            return np.nan, 0.0
        
def pick_brand(row):
    if pd.isna(row["brand_clean"]):
        if row["pair_score"] >= THRESH_PAIR:
            return row["pair_make"]
        else:
            return np.nan
    else:
        return row["brand_clean"]

def pick_model(row):
    if pd.isna(row["model_clean_tmp"]):
        if row["pair_score"] >= THRESH_PAIR:
            return row["pair_model"]
        else:
            return np.nan
    else:
        return row["model_clean_tmp"]

def fuzzy_to_domain(value, domain, conf_min=60):

    match = process.extractOne(value, domain, scorer=fuzz.WRatio)
    if match is None:
        return np.nan
    label, score = match[0], match[1]

    return label

## Load Test Data

In [10]:
df_test = pd.read_csv("../project_data/original_data/test.csv")
df_test.head()

,carID,Brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,89856,Hyundai,I30,2022.878006,Automatic,30700.000000,petrol,205.0,41.5,1.6,61.0,3.0,0.0
1,106581,VW,Tiguan,2017.000000,Semi-Auto,-48190.655673,Petrol,150.0,38.2,2.0,60.0,2.0,0.0
2,80886,BMW,2 Series,2016.000000,Automatic,36792.000000,Petrol,125.0,51.4,1.5,94.0,2.0,0.0
3,100174,Opel,Grandland X,2019.000000,Manual,5533.000000,Petrol,145.0,44.1,1.2,77.0,1.0,0.0
4,81376,BMW,1 Series,2019.000000,Semi-Auto,9058.000000,Diesel,150.0,51.4,2.0,45.0,4.0,0.0


## Preprocess Test Data

In [11]:
car_ids = df_test['carID'].copy()

In [12]:
df_test['Brand'] = df_test['Brand'].str.lower()

In [13]:
df_test['model'] = df_test['model'].str.lower().str.replace('-', ' ')

In [14]:
df_test['model'] = df_test['model'].apply(remove_leading_space)

In [15]:
catalog = pd.read_csv('../project_data/make_model_catalog.csv')

In [16]:
catalog['make'] = catalog['make'].str.lower().str.strip()
catalog['model'] = catalog['model'].str.lower().str.replace('-', ' ')
catalog['model'] = catalog['model'].apply(remove_leading_space)

In [17]:
cat = catalog.copy()
df = df_test.copy()

#### Match and map of car brand

In [18]:
makes_norm = cat["make"].unique().tolist()
brand_uniques = df["Brand"].unique().tolist()

In [19]:
brand_map = {}
for br in brand_uniques:
    best, sc = rf_top1(br, makes_norm, scorer=fuzz.ratio)
    brand_map[br] = {'match':best, 'score': sc}

In [20]:
THRESH_BRAND = 75
filtered_map = {
    key: nested_dict['match']
    for key, nested_dict in brand_map.items()
    if nested_dict['score'] >= THRESH_BRAND
}
filtered_map['v'] = 'vw'

In [21]:
df["brand_clean"] = df['Brand'].map(filtered_map)

#### Match and map car model within his brand

In [22]:
models_by_make = {
    mk: (
        sorted(g["model"].unique().tolist())    )
    for mk, g in cat.groupby("make")
}

mod_by_fil_make = [
    sorted(g["model"].unique().tolist())
    for mk, g in cat.groupby("make")
    if mk in list(set(filtered_map.values()))
]
mod_by_fil_make = [item for sublist in mod_by_fil_make for item in sublist]

In [23]:
THRESH_MODEL = 70.0
THRESH_MODEL_NO_BRAND = 80
tmp = df.apply(lambda r: match_model_in_make(r["model"], r["brand_clean"], r["Brand"]), axis=1)

In [24]:
df["model_clean_tmp"] = [t[0] for t in tmp]
df["model_score_inmake"] = [t[1] for t in tmp]

#### Match and map pairs brand-model when brand alone was not sufficient

In [25]:
pairs_df = cat[["make","model"]].copy()
def fallback_pair(row):
    if not pd.isna(row["brand_clean"]) or pd.isna(row["Brand"]) or pd.isna(row["model"]):
        return np.nan, np.nan, 0.0
    mk_cands = [m for (m,_,_) in process.extract(row["Brand"], makes_norm, scorer=fuzz.ratio, limit=5)]
    if not mk_cands:
        return np.nan, np.nan, 0.0
    sub = pairs_df[pairs_df["make"].isin(mk_cands)].copy()
    q = (row["Brand"] + " " + row["model"])
    choices = (sub["make"] + " " + sub["model"]).tolist()
    best_pair, score = rf_top1(q, choices, scorer=fuzz.ratio)
    hit = sub.loc[(sub["make"] + " " + sub["model"])==best_pair].iloc[0]
    return hit['make'], hit['model'], score

fb = df.apply(fallback_pair, axis=1, result_type="expand")

df["pair_make"], df["pair_model"], df["pair_score"] = fb[0], fb[1], fb[2]

In [26]:
THRESH_PAIR = 65

In [27]:
df["brand_final"] = df.apply(pick_brand, axis=1)
df["model_final"] = df.apply(pick_model, axis=1)

### Brand mapping by model

Since we can map a brand to a specific model we used this to input some of the nan values in the brand variable

In [28]:
MIN_SUPPORT = 2          

gb = df.groupby("model_final")
n_brands_by_model = gb["brand_final"].nunique()

count_by_pair = df.groupby(["model_final","brand_final"]).size().sort_values(ascending=False)

non_ambiguous_models = n_brands_by_model[n_brands_by_model == 1].index

In [29]:
top_brand_per_model = (
    count_by_pair.reset_index().rename(columns={0:"n"})
    .loc[lambda d: d["model_final"].isin(non_ambiguous_models)]
)

In [30]:
top_brand_per_model = top_brand_per_model.loc[top_brand_per_model['n'] >= MIN_SUPPORT]
model2brand_key = dict(zip(top_brand_per_model["model_final"], top_brand_per_model["brand_final"]))

In [31]:
df.loc[df['Brand'].isna(), "brand_final"] = df.loc[df['Brand'].isna(), "model_final"].map(model2brand_key)

In [32]:
df_test = df[['carID', 'brand_final', 'model_final', 'year', 'transmission', 'mileage', 'fuelType', 'tax', 'mpg', 'engineSize','paintQuality%', 'previousOwners','hasDamage']].rename(columns={'brand_final':'brand', 'model_final':'model'}).copy()
df_test

,carID,brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,89856,hyundai,i30,2022.878006,Automatic,30700.000000,petrol,205.0,41.5,1.6,61.0,3.0,0.0
1,106581,vw,tiguan,2017.000000,Semi-Auto,-48190.655673,Petrol,150.0,38.2,2.0,60.0,2.0,0.0
2,80886,bmw,2 series,2016.000000,Automatic,36792.000000,Petrol,125.0,51.4,1.5,94.0,2.0,0.0
3,100174,opel,grandland x,2019.000000,Manual,5533.000000,Petrol,145.0,44.1,1.2,77.0,1.0,0.0
4,81376,bmw,1 series,2019.000000,Semi-Auto,9058.000000,Diesel,150.0,51.4,2.0,45.0,4.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
32562,105775,vw,tiguan,2017.000000,Manual,27575.000000,Petrol,145.0,46.3,1.4,94.0,1.0,0.0
32563,81363,bmw,x2,2020.000000,Automatic,1980.000000,Petrol,145.0,34.0,2.0,39.0,3.0,0.0
32564,76833,audi,q5,2019.000000,Semi-Auto,8297.000000,Diesel,145.0,38.2,2.0,88.0,4.0,0.0
32565,91768,mercedes,a class,2019.000000,Manual,-50755.210230,Petrol,145.0,28.5,1.3,81.0,1.0,0.0


In [33]:
df_test['year'] = df_test['year'].apply(lambda x: int(round(x)) if pd.notnull(x) else np.nan)

In [34]:
df_test['year'] = df_test['year'].astype('Int64')

In [35]:
df_test.loc[df_test['year'] > 2020, 'year'] = np.nan

In [36]:
# Convert mileage to nearest integer.
df_test = df_test.copy()
df_test['mileage'] = df_test['mileage'].round()

In [37]:
# And change dtype to int64.
df_test['mileage'] = df_test['mileage'].astype('Int64')

In [38]:
# Replacing the negative mileage values with NaNs.
df_test.loc[df_test['mileage'] < 0, 'mileage'] = np.nan 

In [39]:
# Replace negative tax values with NaN.
df_test.loc[df_test['tax'] < 0, 'tax'] = np.nan

In [40]:
# Convert all string values in the 'fuelType' column to lowercase.
df_test['fuelType'] = df_test['fuelType'].str.lower()

In [41]:
fuel_values = ["diesel", "petrol", "hybrid","electric", "other"]
df_test["fuelType"] = df_test["fuelType"].apply(lambda x: fuzzy_to_domain(x, domain=fuel_values))

In [42]:
# Replace all 'other' values in the 'fueltype' column with NaN
df_test['fuelType'] = df_test['fuelType'].replace('other', np.nan)

In [43]:
# Round all values in the 'mpg' column to one decimal place
df_test['mpg'] = df_test['mpg'].round(1)

In [44]:
# Round all values in the 'mpg' column to one decimal place
df_test['mpg'] = df_test['mpg'].round(1)

In [45]:
# Rounding all values to just one decimal place.
df_test['engineSize'] = df_test['engineSize'].round(1)

In [46]:
df_test.loc[df_test['engineSize'] <= 0.5, 'engineSize'] = np.nan

In [47]:
# Round to nearest integer.
df_test['previousOwners'] = df_test['previousOwners'].round(0)

In [48]:
# Replace negative values with NaN.
df_test['previousOwners'] = df_test['previousOwners'].apply(lambda x: np.nan if x < 0 else x)

In [49]:
# Convert the column to integer type (Int64 allows NaN values).
df_test['previousOwners'] = df_test['previousOwners'].astype('Int64')

In [50]:
# Remove leading/trailing spaces and lowercase all entries.
df_test['transmission'] = df_test['transmission'].str.strip().str.lower()

In [51]:
transmission_values = ["automatic", "manual", "semi-auto", "other", "unknown"]
df_test["transmission"] = df_test["transmission"].apply(lambda x: fuzzy_to_domain(x, domain=transmission_values))

In [52]:
# Replace all 'unknown' and 'other' values in the 'transmission' column with NaN
df_test['transmission'] = df_test['transmission'].replace('unknown', np.nan)
df_test['transmission'] = df_test['transmission'].replace('other', np.nan)

In [53]:
df_test["car_age"] = 2020 - df_test["year"]

In [54]:
df_test = df_test.drop(columns=['year'])

In [55]:
df_test = df_test.drop(columns=['carID', 'previousOwners', 'paintQuality%', 'hasDamage'])

## Load Model and Generate Predictions

In [56]:
model = joblib.load("../project_data/models/final_model.pkl")

In [57]:
predictions = model.predict(df_test)

In [58]:
df_test

,brand,model,transmission,mileage,fuelType,tax,mpg,engineSize,car_age
0,hyundai,i30,automatic,30700,petrol,205.0,41.5,1.6,<NA>
1,vw,tiguan,semi-auto,<NA>,petrol,150.0,38.2,2.0,3
2,bmw,2 series,automatic,36792,petrol,125.0,51.4,1.5,4
3,opel,grandland x,manual,5533,petrol,145.0,44.1,1.2,1
4,bmw,1 series,semi-auto,9058,diesel,150.0,51.4,2.0,1
...,...,...,...,...,...,...,...,...,...
32562,vw,tiguan,manual,27575,petrol,145.0,46.3,1.4,3
32563,bmw,x2,automatic,1980,petrol,145.0,34.0,2.0,0
32564,audi,q5,semi-auto,8297,diesel,145.0,38.2,2.0,1
32565,mercedes,a class,manual,<NA>,petrol,145.0,28.5,1.3,1


## Create Submission

In [59]:
submission_df = pd.DataFrame({'carID': car_ids, 'price': predictions})
submission_df.to_csv('Group60_Version_02.csv', index=False)
submission_df.head()

,carID,price
0,89856,14362.814812
1,106581,24556.662588
2,80886,13815.065503
3,100174,16701.784420
4,81376,24839.045467
